In [1]:
!unzip -o archive.zip

Archive:  archive.zip
  inflating: online_retail.csv       


In [2]:
import pandas as pd

# ---------- 1. LOAD + CLEAN ----------
df = pd.read_csv("online_retail.csv")
print("Raw rows:", len(df), "(must be 541909)")
df = df.dropna(subset=["InvoiceDate"])
dups = df.duplicated().sum()
df = df.drop_duplicates()
print("Duplicates removed:", dups, "| Rows now:", len(df))

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)
df["Quantity"] = df["Quantity"].astype(int)
df["CustomerID"] = df["CustomerID"].astype("Int64")
df["Description"] = df["Description"].str.strip().str.upper()

# phantom orders: >£5,000 order fully cancelled by same customer, same product, within 48h
known = df[df["CustomerID"].notna()]
orders = known[(~known["InvoiceNo"].str.startswith("C")) & (known["Quantity"] > 0)
               & (known["Quantity"] * known["UnitPrice"] > 5000)]
cancs = known[known["InvoiceNo"].str.startswith("C")]
o = orders.reset_index().rename(columns={"index": "o_idx"})
c = cancs.reset_index().rename(columns={"index": "c_idx"})
c["Quantity"] = -c["Quantity"]
m = o.merge(c[["c_idx", "CustomerID", "StockCode", "Quantity", "InvoiceDate"]],
            on=["CustomerID", "StockCode", "Quantity"], suffixes=("", "_c"))
m["gap_hours"] = (m["InvoiceDate_c"] - m["InvoiceDate"]).dt.total_seconds() / 3600
m = m[(m["gap_hours"] >= 0) & (m["gap_hours"] <= 48)].drop_duplicates("o_idx").drop_duplicates("c_idx")
m["Value"] = m["Quantity"] * m["UnitPrice"]
print("\nPhantom orders:")
print(m[["InvoiceNo", "Description", "Quantity", "Value", "gap_hours"]].round(2).to_string(index=False))
phantom = df.loc[list(m["o_idx"]) + list(m["c_idx"])].copy()
df = df.drop(phantom.index)

# fees / non-products
nonprod = ["POST", "DOT", "M", "m", "C2", "D", "S", "B", "AMAZONFEE", "BANK CHARGES", "CRUK"]
is_fee = df["StockCode"].isin(nonprod) | df["StockCode"].str.startswith("gift_")
fees = df[is_fee].copy()
prod = df[~is_fee]

# cancellations / adjustments / sales
is_cancel = prod["InvoiceNo"].str.startswith("C")
cancellations = prod[is_cancel].copy()
rest = prod[~is_cancel]
adjustments = rest[(rest["Quantity"] <= 0) | (rest["UnitPrice"] <= 0)].copy()
sales = rest[(rest["Quantity"] > 0) & (rest["UnitPrice"] > 0)].copy()

# revenue + time columns
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
cancellations["Revenue"] = cancellations["Quantity"] * cancellations["UnitPrice"]
for d in [sales, cancellations]:
    d["Date"] = d["InvoiceDate"].dt.normalize()
    d["Year"] = d["InvoiceDate"].dt.year
    d["Month"] = d["InvoiceDate"].dt.month
    d["DayOfWeek"] = d["InvoiceDate"].dt.day_name()
    d["Hour"] = d["InvoiceDate"].dt.hour
sales["HasCustomer"] = sales["CustomerID"].notna()

print("\nSales:", len(sales), "| Cancellations:", len(cancellations), "| Adjustments:", len(adjustments),
      "| Fees:", len(fees), "| Phantom:", len(phantom))
print("Total:", len(sales) + len(cancellations) + len(adjustments) + len(fees) + len(phantom),
      "| Should equal:", len(df) + len(phantom))
print("Gross revenue: £", round(sales["Revenue"].sum()))

Raw rows: 541909 (must be 541909)
Duplicates removed: 5268 | Rows now: 536641

Phantom orders:
InvoiceNo                    Description  Quantity    Value  gap_hours
   541431 MEDIUM CERAMIC TOP STORAGE JAR     74215  77183.6       0.27
   556444 PICNIC BASKET WICKER 60 PIECES        60  38970.0       0.18
   581483    PAPER CRAFT , LITTLE BIRDIE     80995 168469.6       0.20

Sales: 522537 | Cancellations: 8665 | Adjustments: 2493 | Fees: 2940 | Phantom: 6
Total: 536641 | Should equal: 536641
Gross revenue: £ 9962596


In [3]:
import re

patches = [
    ("Bags, Boxes & Storage", ["JUMBO BAG"]),
    ("Toys, Games & Crafts", ["CARD GAME", "SNAKES & LADDERS", "LUDO", "BINGO", "SPINNING TOP", "FELTCRAFT", "BABY GIFT SET"]),
    ("Jewellery & Accessories", ["LIP GLOSS", "PONCHO", "SOMBRERO"]),
    ("Kitchen & Dining", ["BREAKFAST SET", "BEAKER", "PLACEMAT", "FRYING PAN", "TABLECLOTH"]),
    ("Home Decor & Garden", ["ACAPULCO MAT", "MAGNET", "TRELLIS", "LOVE SEAT", "DRAWERKNOB", "WICKER HEART", "FILIGRIS",
                             "TRINKET POT", "ROSETTE", "JEWELLERY DRAWER", "JEWELLERY STAND"]),
    ("Candles & Lighting", ["TORCH"]),
]
priority = [
    ("Bags, Boxes & Storage", ["LUNCH BAG"]),
    ("Home Decor & Garden",   ["CARD HOLDER", "BOOK STAND", "HEART OF WICKER"]),
    ("Kitchen & Dining",      ["FOOD COVER", "PICNIC BASKET", "PICNIC"]),
]
rules = [
    ("Candles & Lighting", ["CANDLE", "CANDLEHOLDER", "CHOCOLATECANDLE", "LIGHT", "TLIGHT", "TEALIGHT",
                            "LANTERN", "LAMP", "LAMPSHADE", "NIGHTLIGHT"]),
    ("Cards, Wrap & Stationery", ["CARD", "WRAP", "WRAPPING", "TAG", "STICKER", "NOTEBOOK", "PENCIL", "PEN",
                                  "RIBBON", "BOOK", "ENVELOPE", "STATIONERY", "CALENDAR", "JOURNAL", "ERASER",
                                  "RUBBER", "RULER", "NOTEPAD", "DIARY", "TISSUE"]),
    ("Jewellery & Accessories", ["NECKLACE", "BRACELET", "EARRING", "RING", "BEAD", "JEWELLERY", "JEWELRY",
                                 "SCARF", "PURSE", "WALLET", "SUNGLASSES", "HAIRGRIP", "HAIRCLIP", "HAIRBAND",
                                 "BROOCH", "PENDANT", "CHARM", "BANGLE", "UMBRELLA", "SLIPPER", "GLOVES",
                                 "GLOSS", "LIP GLOSS", "FAN"]),
    ("Toys, Games & Crafts", ["TOY", "GAME", "DOLL", "PUZZLE", "KIDS", "TEDDY", "SKITTLES", "PLAYHOUSE",
                              "CRAYON", "DOMINOES", "JIGSAW", "BALL", "ROBOT", "BUILDING BLOCK", "BLOCK",
                              "PAINT", "CLAY", "COLOURING", "KNITTING", "SEWING", "GLIDERS", "SKIPPING",
                              "PIGGY BANK"]),
    ("Home Decor & Garden", ["HOT WATER BOTTLE", "HOTTIE", "HAND WARMER", "FRAME", "MIRROR", "CLOCK", "CUSHION",
                             "COVER", "DOORMAT", "DOORSTOP", "HOOK", "SIGN", "WALL", "VASE", "FLOWER", "PHOTO",
                             "DOOR", "SHELF", "BLANKET", "CURTAIN", "RUG", "ORNAMENT", "HANGER", "GARDEN",
                             "GARDENERS", "KNEELING", "PLANT", "SEED", "BIRD", "WINDMILL", "FEEDER", "PARASOL",
                             "WATERING", "HAMMOCK", "BOARD", "MEMOBOARD", "CHALKBOARD", "BLACKBOARD", "PEGS",
                             "FILIGREE", "PEARS"]),
    ("Kitchen & Dining", ["MUG", "CUP", "TEAPOT", "TEA", "PLATE", "BOWL", "CAKE", "CAKESTAND", "SPOON", "CUTLERY",
                          "GLASS", "GLASSES", "JUG", "BOTTLE", "BAKING", "JAM", "LUNCH", "NAPKIN", "APRON",
                          "SAUCER", "KITCHEN", "DISH", "COASTER", "TRAY", "KNIFE", "FORK", "COOKIE", "COFFEE",
                          "SUGAR", "BREAD", "EGG", "TOWEL", "OVEN", "CHOPPING", "POPCORN", "PANTRY", "JELLY",
                          "MOULD", "ROLLING PIN", "DOILIES", "PLACE SETTING"]),
    ("Bags, Boxes & Storage", ["BAG", "BASKET", "BOX", "BOXES", "TIN", "JAR", "STORAGE", "CASE", "TRUNK",
                               "CRATE", "ORGANISER", "ORGANIZER", "RACK", "DRAWER", "CABINET", "CHEST",
                               "POUCH", "TOTE", "SHOPPER", "BACKPACK", "CONTAINER"]),
    ("Party & Seasonal", ["CHRISTMAS", "XMAS", "PARTY", "BIRTHDAY", "EASTER", "HALLOWEEN", "BUNTING", "BALLOON",
                          "SANTA", "SNOW", "SNOWFLAKE", "ADVENT", "TREE", "DECORATION", "GARLAND", "WEDDING",
                          "VALENTINE", "PAPER CHAIN"]),
]

def make_pattern(words):
    return re.compile(r"\b(?:" + "|".join(re.escape(w) + "S?" for w in words) + r")\b")

compiled = [(cat, make_pattern(w)) for cat, w in patches + priority + rules]

def categorize(desc):
    for cat, pat in compiled:
        if pat.search(desc):
            return cat
    return "Other"

cat_map = {d: categorize(d) for d in sales["Description"].unique()}
sales["Category"] = sales["Description"].map(cat_map)

summary = (sales.groupby("Category")
           .agg(Revenue=("Revenue", "sum"), Units=("Quantity", "sum"), Products=("Description", "nunique"))
           .sort_values("Revenue", ascending=False))
summary["Share_%"] = (summary["Revenue"] / summary["Revenue"].sum() * 100).round(1)
summary["Revenue"] = summary["Revenue"].round(0)
print("Unique products:", len(cat_map), "| Total:", summary["Revenue"].sum())
summary

Unique products: 4001 | Total: 9962595.0


,Revenue,Units,Products,Share_%
Category,,,,
Home Decor & Garden,2484471.0,879481,808,24.9
Kitchen & Dining,2290609.0,1140768,676,23.0
"Bags, Boxes & Storage",1865539.0,921896,350,18.7
Candles & Lighting,886433.0,477151,339,8.9
Party & Seasonal,694156.0,372816,240,7.0
"Cards, Wrap & Stationery",602190.0,810331,473,6.0
"Toys, Games & Crafts",553916.0,319496,149,5.6
Other,400931.0,329624,539,4.0
Jewellery & Accessories,184350.0,154554,427,1.9


In [4]:
# ---------- KPIs ----------
gross = sales["Revenue"].sum()
cancelled = cancellations["Revenue"].sum()
orders = sales["InvoiceNo"].nunique()
print("Gross £", round(gross), "| Cancelled £", round(cancelled), "| Net £", round(gross + cancelled))
print("Cancellation rate by value:", round(-cancelled / gross * 100, 2), "%")
print("Orders:", orders, "| Customers:", sales["CustomerID"].nunique(), "| Units:", sales["Quantity"].sum())
print("Average order value: £", round(gross / orders, 2))

m_sales = sales.groupby(sales["InvoiceDate"].dt.to_period("M"))["Revenue"].sum()
m_cancel = cancellations.groupby(cancellations["InvoiceDate"].dt.to_period("M"))["Revenue"].sum()
monthly = pd.DataFrame({"Gross": m_sales, "Cancelled": m_cancel}).fillna(0)
monthly["Net"] = monthly["Gross"] + monthly["Cancelled"]
monthly["MoM_%"] = (monthly["Gross"].pct_change() * 100).round(1)
monthly["Cancel_%"] = (-monthly["Cancelled"] / monthly["Gross"] * 100).round(1)
monthly[["Gross", "Cancelled", "Net"]] = monthly[["Gross", "Cancelled", "Net"]].round(0)
print("\n", monthly)

prod_t = (sales.groupby("Description")
          .agg(Revenue=("Revenue", "sum"), Units=("Quantity", "sum"), Category=("Category", "first")))
prod_t["Share_%"] = (prod_t["Revenue"] / prod_t["Revenue"].sum() * 100).round(2)
prod_t["Revenue"] = prod_t["Revenue"].round(0)
top_rev = prod_t.sort_values("Revenue", ascending=False).head(10)
top_units = prod_t.sort_values("Units", ascending=False).head(10)
print("\nTOP 10 BY REVENUE\n", top_rev)
print("\nTOP 10 BY UNITS\n", top_units)

country = (sales.groupby("Country")
           .agg(Revenue=("Revenue", "sum"), Orders=("InvoiceNo", "nunique"), Customers=("CustomerID", "nunique"))
           .sort_values("Revenue", ascending=False))
country["Share_%"] = (country["Revenue"] / country["Revenue"].sum() * 100).round(1)
country["Revenue"] = country["Revenue"].round(0)
print("\nCountries:", len(country)); print(country.head(10))

# ---------- SAVE EVERYTHING AS ONE ZIP ----------
import zipfile, os
from google.colab import files
files_out = {
    "clean_sales_categorized.csv": sales, "cancellations.csv": cancellations, "adjustments.csv": adjustments,
    "fees.csv": fees, "phantom_orders.csv": phantom,
    "product_category_map.csv": pd.Series(cat_map, name="Category").rename_axis("Description").reset_index(),
    "kpi_category.csv": summary.reset_index(), "kpi_monthly.csv": monthly.reset_index(),
    "kpi_top_products_revenue.csv": top_rev.reset_index(), "kpi_top_products_units.csv": top_units.reset_index(),
    "kpi_country.csv": country.reset_index(),
}
with zipfile.ZipFile("FUTURE_DS_01_data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name, data in files_out.items():
        data.to_csv(name, index=False)
        z.write(name)
print("\nZip size (MB):", round(os.path.getsize("FUTURE_DS_01_data.zip") / 1e6, 1))
files.download("FUTURE_DS_01_data.zip")

Gross £ 9962596 | Cancelled £ -229951 | Net £ 9732645
Cancellation rate by value: 2.31 %
Orders: 19770 | Customers: 4333 | Units: 5406117
Average order value: £ 503.92

                  Gross  Cancelled        Net  MoM_%  Cancel_%
InvoiceDate                                                  
2010-12       775638.0   -17548.0   758090.0    NaN       2.3
2011-01       593197.0   -14341.0   578855.0  -23.5       2.4
2011-02       507800.0    -8336.0   499464.0  -14.4       1.6
2011-03       689834.0   -10504.0   679329.0   35.8       1.5
2011-04       515378.0   -33316.0   482062.0  -25.3       6.5
2011-05       739953.0    -8948.0   731005.0   43.6       1.2
2011-06       698589.0   -13417.0   685172.0   -5.6       1.9
2011-07       688219.0   -11407.0   676812.0   -1.5       1.7
2011-08       724216.0   -22927.0   701290.0    5.2       3.2
2011-09      1028320.0   -16981.0  1011340.0   42.0       1.7
2011-10      1103314.0   -41613.0  1061701.0    7.3       3.8
2011-11      1452116.0  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>